In [1]:
import os
import time
import random
import statistics

import numpy as np
import torch

from torchvision.models import mobilenet_v2

from pytorchexample.task import load_data, train

print("Imports successful.")
print("PyTorch:", torch.__version__)

f:\Flower\quickstart-pytorch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful.
PyTorch: 2.10.0+cpu


In [4]:
# Experiment configuration
CLIENT_ID = 0
NUM_CLIENTS = 5
BATCH_SIZE = 32
LOCAL_EPOCHS = 1
LEARNING_RATE = 0.01

# Load the actual Flower client partition
trainloader, valloader = load_data(
    partition_id=CLIENT_ID,
    num_partitions=NUM_CLIENTS,
    batch_size=BATCH_SIZE,
)

print("========== CLIENT DATA ==========")

print("Client ID:", CLIENT_ID)
print("Training samples:", len(trainloader.dataset))
print("Training batches:", len(trainloader))
print("Validation samples:", len(valloader.dataset))
print("Validation batches:", len(valloader))

print("Batch size:", BATCH_SIZE)
print("Local epochs:", LOCAL_EPOCHS)

========== CLIENT DATA ==========
Client ID: 0
Training samples: 2275
Training batches: 72
Validation samples: 569
Validation batches: 18
Batch size: 32
Local epochs: 1


In [5]:
# Select the available device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configure CPU threads when using CPU
if DEVICE.type == "cpu":
    torch.set_num_threads(2)

# Model configuration
NUM_CLASSES = 10

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Create the model
net = mobilenet_v2(
    weights=None,
    num_classes=NUM_CLASSES,
)

num_parameters = sum(
    p.numel() for p in net.parameters() if p.requires_grad
)

print("Model: MobileNetV2")
print("Trainable parameters:", num_parameters)

Device: cpu
Model: MobileNetV2
Trainable parameters: 2236682


In [7]:
def synchronize_device():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def run_client_training():
    # Fresh model for every run
    model = mobilenet_v2(
        weights=None,
        num_classes=NUM_CLASSES,
    )

    # Use the same training function as Flower
    start = time.perf_counter()

    train_loss = train(
        net=model,
        trainloader=trainloader,
        epochs=LOCAL_EPOCHS,
        lr=LEARNING_RATE,
        device=DEVICE,
    )

    synchronize_device()
    elapsed = time.perf_counter() - start

    return elapsed, train_loss


# Warm-up: one complete local epoch
print("Running warm-up...")
warmup_time, warmup_loss = run_client_training()

print("Warm-up completed.")
print("Warm-up duration:", round(warmup_time, 3), "seconds")
print("Warm-up loss:", round(warmup_loss, 4))

Running warm-up...
Warm-up completed.
Warm-up duration: 43.988 seconds
Warm-up loss: 1.3546


In [8]:
NUM_TRIALS = 3

trial_times = []
trial_losses = []

for trial in range(NUM_TRIALS):
    elapsed, loss = run_client_training()

    trial_times.append(elapsed)
    trial_losses.append(loss)

    print(
        f"Trial {trial + 1}: "
        f"{elapsed:.3f} seconds | "
        f"Loss: {loss:.4f}"
    )


median_time = statistics.median(trial_times)
q1, q3 = statistics.quantiles(
    trial_times,
    n=4,
    method="inclusive",
)[0::2]

iqr = q3 - q1

total_images = len(trainloader.dataset) * LOCAL_EPOCHS
throughput = total_images / median_time
time_per_batch = median_time / (len(trainloader) * LOCAL_EPOCHS)

print("\n========== CLIENT TRAINING BENCHMARK ==========")
print("Client:", CLIENT_ID)
print("Device:", DEVICE)
print("Training samples:", len(trainloader.dataset))
print("Training batches:", len(trainloader))
print("Local epochs:", LOCAL_EPOCHS)

print("\nTrial times:", [round(t, 3) for t in trial_times])
print("Median duration:", round(median_time, 3), "seconds")
print("Q1:", round(q1, 3), "seconds")
print("Q3:", round(q3, 3), "seconds")
print("IQR:", round(iqr, 3), "seconds")

print("\nTime per batch:", round(time_per_batch, 4), "seconds")
print("Training throughput:", round(throughput, 2), "images/second")

Trial 1: 108.116 seconds | Loss: 1.2563
Trial 2: 68.662 seconds | Loss: 1.3024
Trial 3: 80.263 seconds | Loss: 1.3641

========== CLIENT TRAINING BENCHMARK ==========
Client: 0
Device: cpu
Training samples: 2275
Training batches: 72
Local epochs: 1

Trial times: [108.116, 68.662, 80.263]
Median duration: 80.263 seconds
Q1: 74.463 seconds
Q3: 94.19 seconds
IQR: 19.727 seconds

Time per batch: 1.1148 seconds
Training throughput: 28.34 images/second


In [9]:
import hashlib

def dataset_fingerprint(dataset):
    fingerprint = hashlib.sha256()

    for i in range(len(dataset)):
        sample = dataset[i]

        image = sample["image"].contiguous().cpu().numpy()
        label = int(sample["label"])

        fingerprint.update(label.to_bytes(4, "little", signed=True))
        fingerprint.update(image.tobytes())

    return fingerprint.hexdigest()


train_hash = dataset_fingerprint(trainloader.dataset)
val_hash = dataset_fingerprint(valloader.dataset)

print("Training fingerprint:", train_hash)
print("Validation fingerprint:", val_hash)

Training fingerprint: ff625b9393c6262c3bac63cba4acdc14ef116e7996e2a8aaffad6f80fbb30197
Validation fingerprint: 3eaf832b66d0414a0f5095dfac605b438fb683962f16391cf6247a678fae3349
